# CEG-WM Colab 完整工作流执行指南

本 Notebook 在 Google Colab 上执行完整的机器学习工作流（embed/detect/calibrate/evaluate），生成新的 run_root 目录并下载最终结果。

**执行流水线**：
- **Embed 阶段**：对输入进行特征提取和防水印嵌入
- **Detect 阶段**：检测防水印信号
- **Calibrate 阶段**：校准检测阈值和概率
- **Evaluate 阶段**：评估整个工作流的性能

## 第 1 步：清理旧代码并拉取 GitHub 项目

从 GitHub 拉取 CEG-WM 代码到 Colab 工作目录。

**方式：自动拉取指定仓库与分支**
- 仓库：https://github.com/RICHAAARC/CEG-WM.git
- 分支：main

In [ ]:
import sys
import os
from pathlib import Path
import psutil

# 检测 Google Colab 环境
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("=" * 80)
print("初始化 Notebook 环境")
print("=" * 80)

# 设置工作目录（按需固定到 /content，避免 /tmp 被系统回收）
if IN_COLAB:
    WORK_DIR = Path("/content")
    print(f"✓ 检测到 Google Colab 环境")
else:
    WORK_DIR = Path.cwd()
    print(f"✓ 本地 Jupyter 环境")

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"  工作目录：{WORK_DIR}")

# 显示系统信息
print("\n系统信息：")
print(f"  操作系统：{sys.platform}")
print(f"  Python 版本：{sys.version.split()[0]}")
print(f"  CPU 核心数：{psutil.cpu_count()}")
print(f"  总内存：{psutil.virtual_memory().total / (1024**3):.1f} GB")
print(f"  可用内存：{psutil.virtual_memory().available / (1024**3):.1f} GB")

print("\n✓ 环境初始化完成")

In [ ]:
from pathlib import Path

print("=" * 80)
print("挂载 Google Drive")
print("=" * 80)

DRIVE_MOUNT_POINT = Path("/content/drive")
GOOGLE_DRIVE_ROOT = None
GOOGLE_DRIVE_EXPORT_DIR = None

if IN_COLAB:
    try:
        from google.colab import drive

        drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)
        GOOGLE_DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
        GOOGLE_DRIVE_EXPORT_DIR = GOOGLE_DRIVE_ROOT / "CEG_WM_Outputs" / "Paper_Full_Cuda"
        GOOGLE_DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"✓ Google Drive 已挂载：{GOOGLE_DRIVE_ROOT}")
        print(f"✓ 导出同步目录：{GOOGLE_DRIVE_EXPORT_DIR}")
    except Exception as exc:
        GOOGLE_DRIVE_ROOT = None
        GOOGLE_DRIVE_EXPORT_DIR = None
        print(f"✗ Google Drive 挂载失败：{exc}")
else:
    print("ℹ 当前不是 Google Colab 环境，跳过 Google Drive 挂载")

In [ ]:
import json
import shutil
import subprocess
from datetime import datetime
from pathlib import Path

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "main"
TARGET_DIR = WORK_DIR / "CEG-WM"

print("=" * 80)
print("同步 GitHub 上的 CEG-WM 代码并清理 outputs")
print("=" * 80)
print(f"目标仓库：{REPO_URL}")
print(f"目标分支：{REPO_BRANCH}")
print(f"工作目录：{WORK_DIR}")

if shutil.which("git") is None:
    raise RuntimeError("当前环境未检测到 git 命令，请先安装 git 后重试")

WORK_DIR.mkdir(parents=True, exist_ok=True)


def _print_command_tail(label: str, text: str, max_lines: int = 20) -> None:
    print(f"\n{label}（最后 {max_lines} 行）：")
    lines = (text or "").splitlines()
    print("\n".join(lines[-max_lines:]) if lines else "<empty>")


def _run_checked_command(command, cwd=None):
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    if result.returncode != 0:
        _print_command_tail("STDOUT", result.stdout)
        _print_command_tail("STDERR", result.stderr)
        raise RuntimeError(
            f"命令执行失败，return_code={result.returncode}，command={' '.join(command)}"
        )
    return result


sync_mode = "git_clone"
remote_url = ""
repo_exists = TARGET_DIR.exists() and (TARGET_DIR / ".git").exists()

print("\n[1/4] 同步仓库代码...")
if repo_exists:
    remote_result = subprocess.run(
        ["git", "-C", str(TARGET_DIR), "remote", "get-url", "origin"],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    remote_url = remote_result.stdout.strip() if remote_result.returncode == 0 else ""
    if remote_url and remote_url.rstrip("/") != REPO_URL.rstrip("/"):
        print(f"  检测到 origin 不匹配，删除后重新拉取：{remote_url}")
        shutil.rmtree(TARGET_DIR)
        repo_exists = False

if repo_exists:
    sync_mode = "git_fetch_reset"
    print(f"  复用现有仓库：{TARGET_DIR}")
    _run_checked_command(["git", "-C", str(TARGET_DIR), "fetch", "origin", REPO_BRANCH, "--depth", "1"])
    _run_checked_command(["git", "-C", str(TARGET_DIR), "checkout", "-B", REPO_BRANCH, "FETCH_HEAD"])
    _run_checked_command(["git", "-C", str(TARGET_DIR), "reset", "--hard", "FETCH_HEAD"])
    _run_checked_command(["git", "-C", str(TARGET_DIR), "clean", "-fd"])
    print("  ✓ 仓库已更新到远端最新提交")
else:
    if TARGET_DIR.exists():
        print(f"  检测到非 Git 目录，删除后重新拉取：{TARGET_DIR}")
        shutil.rmtree(TARGET_DIR)

    clone_cmd = [
        "git", "clone",
        "--depth", "1",
        "--branch", REPO_BRANCH,
        REPO_URL,
        str(TARGET_DIR),
    ]
    print("  执行命令：")
    print("    " + " ".join(clone_cmd))
    _run_checked_command(clone_cmd)
    print(f"  ✓ 首次拉取完成：{TARGET_DIR}")

print("\n[2/4] 清理当前 outputs...")
outputs_dir = TARGET_DIR / "outputs"
if outputs_dir.exists():
    shutil.rmtree(outputs_dir)
    print(f"  ✓ 已删除旧输出目录：{outputs_dir}")
else:
    print("  ✓ 未发现旧 outputs 目录")
outputs_dir.mkdir(parents=True, exist_ok=True)
print(f"  ✓ 已重建 outputs 目录：{outputs_dir}")

print("\n[3/4] 校验关键目录结构...")
required_paths = [
    TARGET_DIR / "main" / "cli",
    TARGET_DIR / "configs",
    TARGET_DIR / "scripts",
    TARGET_DIR / "requirements.txt",
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"同步完成但目录结构不完整，缺失：{missing_paths}")
print("  ✓ 关键目录结构校验通过")

print("\n[4/4] 记录 Git 版本信息...")
GIT_INFO_PATH = WORK_DIR / "git_version_info.json"

try:
    commit_hash = _run_checked_command([
        "git", "-C", str(TARGET_DIR), "rev-parse", "HEAD"
    ]).stdout.strip()
    short_hash = _run_checked_command([
        "git", "-C", str(TARGET_DIR), "rev-parse", "--short=8", "HEAD"
    ]).stdout.strip()
    commit_message = _run_checked_command([
        "git", "-C", str(TARGET_DIR), "log", "-1", "--pretty=%s"
    ]).stdout.strip()
    commit_timestamp = _run_checked_command([
        "git", "-C", str(TARGET_DIR), "log", "-1", "--pretty=%ci"
    ]).stdout.strip()
    author = _run_checked_command([
        "git", "-C", str(TARGET_DIR), "log", "-1", "--pretty=%an <%ae>"
    ]).stdout.strip()

    version_info = {
        "repo_url": REPO_URL,
        "branch": REPO_BRANCH,
        "target_dir": str(TARGET_DIR),
        "sync_mode": sync_mode,
        "outputs_cleaned": True,
        "commit_hash": commit_hash,
        "commit_hash_short": short_hash,
        "commit_message": commit_message,
        "commit_timestamp": commit_timestamp,
        "commit_author": author,
        "sync_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    with open(GIT_INFO_PATH, "w", encoding="utf-8") as f:
        json.dump(version_info, f, indent=2, ensure_ascii=False)

    print(f"✓ Git 版本信息已记录：{GIT_INFO_PATH}")
    print(f"  Sync mode: {sync_mode}")
    print(f"  Commit: {short_hash} - {commit_message}")
    print(f"  Author: {author}")
    print(f"  时间戳: {commit_timestamp}")
except Exception as exc:
    print(f"⚠ 版本信息记录失败（不影响主流程）：{exc}")
    fallback_info = {
        "repo_url": REPO_URL,
        "branch": REPO_BRANCH,
        "target_dir": str(TARGET_DIR),
        "sync_mode": sync_mode,
        "outputs_cleaned": True,
        "commit_hash": "<获取失败>",
        "sync_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "error": str(exc),
    }
    with open(GIT_INFO_PATH, "w", encoding="utf-8") as f:
        json.dump(fallback_info, f, indent=2, ensure_ascii=False)
    print(f"⚠ 已写入备用 Git 信息文件：{GIT_INFO_PATH}")

In [ ]:

# 验证和定位 CEG-WM 代码库
from pathlib import Path

def find_ceg_wm_root(base_dir):
    """
    功能：查找 CEG-WM 根目录（智能路径发现）。
    """
    base_dir = Path(base_dir)

    # 检查：特征目录结构
    characteristic_dirs = ["main/cli", "configs", "scripts", "requirements.txt"]

    # 首先检查直接目录
    if all((base_dir / d).exists() for d in characteristic_dirs):
        return base_dir

    # 查找包含正确结构的子目录
    for subdir in sorted(base_dir.iterdir()):
        if subdir.is_dir() and not subdir.name.startswith('.'):
            if all((subdir / d).exists() for d in characteristic_dirs):
                return subdir

    # 尝试找任何包含 main/cli 的目录（最后的备选方案）
    for possible_root in base_dir.rglob("main"):
        if possible_root.is_dir():
            ceg_root = possible_root.parent
            if (ceg_root / "configs").exists() and (ceg_root / "scripts").exists():
                return ceg_root

    raise FileNotFoundError(
        f"无法找到 CEG-WM 根目录\n"
        f"  搜索路径：{base_dir}\n"
        f"  期望目录结构：main/cli/, configs/, scripts/, requirements.txt\n"
        f"  请检查 GitHub 拉取步骤是否执行成功"
    )

# 定位 CEG-WM 根目录（GitHub 拉取后）
try:
    CEG_WM_ROOT = find_ceg_wm_root(WORK_DIR)
    print(f"✓ 找到 CEG-WM 根目录：{CEG_WM_ROOT}")
    print(f"  绝对路径：{CEG_WM_ROOT.resolve()}")
except FileNotFoundError as e:
    print(f"✗ {e}")
    # 列出实际的目录结构以帮助调试
    print("\n实际目录结构：")
    for item in WORK_DIR.iterdir():
        if item.is_dir() and not item.name.startswith('.'):
            print(f"  {item.name}/")
            for subitem in item.iterdir():
                if subitem.is_dir():
                    print(f"    {subitem.name}/")


In [ ]:

# 添加 CEG-WM 到 Python 路径
if str(CEG_WM_ROOT) not in sys.path:
    sys.path.insert(0, str(CEG_WM_ROOT))

# 验证关键模块可导入
print("验证关键模块导入...")
try:
    from main.cli import run_embed
    print("  ✓ main.cli.run_embed 导入成功")
except ImportError as e:
    print(f"  ✗ 导入失败：{e}")
    print("    这表示 CEG-WM 依赖未完整安装，将在下一步修复")


## 第 2 步：安装依赖包

安装 CEG-WM 项目本身和所有依赖。这是关键步骤！

In [ ]:

import subprocess
import sys

print("=" * 80)
print("安装依赖包")
print("=" * 80)

# 步骤 1：升级 pip
print("\n[1/4] 升级 pip...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"],
               capture_output=True)
print("  ✓ pip 已升级")

# 步骤 2：安装系统依赖
if IN_COLAB:
    print("\n[2/4] 安装系统依赖...")
    subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git", "wget", "unzip"],
                   capture_output=True)
    print("  ✓ 系统依赖已安装")

# 步骤 3：安装 CEG-WM 项目本身（关键！）
print("\n[3/4] 安装 CEG-WM 项目...")
try:
    # 方式 A：从 pyproject.toml（推荐）
    pyproject_file = CEG_WM_ROOT / "pyproject.toml"
    requirements_file = CEG_WM_ROOT / "requirements.txt"

    if pyproject_file.exists():
        print(f"  从 pyproject.toml 安装：{pyproject_file}")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-e", str(CEG_WM_ROOT)],
            capture_output=True,
            text=True,
            timeout=300
        )
        if result.returncode == 0:
            print("  ✓ 项目安装成功（开发模式）")
        else:
            print(f"  ⚠ 项目安装失败，尝试备选方案")
            print(f"    错误：{result.stderr[-200:]}")

    elif requirements_file.exists():
        print(f"  从 requirements.txt 安装：{requirements_file}")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-r", str(requirements_file)],
            capture_output=True,
            text=True,
            timeout=300
        )
        if result.returncode == 0:
            print("  ✓ 依赖安装成功")
        else:
            print(f"  ⚠ 部分依赖安装失败，继续使用...")
    else:
        print("  ⚠ 未找到 pyproject.toml 或 requirements.txt")

except subprocess.TimeoutExpired:
    print("  ⚠ 安装超时（300 秒），但继续执行")
except Exception as e:
    print(f"  ⚠ 安装出错：{e}，但继续执行")

# 步骤 4：安装 transparent-background（InSPyReNet 语义掩码模型驱动包）
print("\n[4/4] 安装 transparent-background（InSPyReNet 依赖）...")
try:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "transparent-background"],
        capture_output=True,
        text=True,
        timeout=120
    )
    if result.returncode == 0:
        print("  ✓ transparent-background 安装成功")
    else:
        print(f"  ⚠ transparent-background 安装失败：{result.stderr[-200:]}")
        print("    InSPyReNet 语义掩码将以 fallback 模式运行（影响内容感知精度）")
except subprocess.TimeoutExpired:
    print("  ⚠ 安装超时（120 秒），但继续执行")
except Exception as e:
    print(f"  ⚠ 安装出错：{e}，但继续执行")

print("\n✓ 依赖安装步骤完成")


## 第 3 步：下载模型权重

真实下载 Stable Diffusion 3 模型。这是关键步骤！

In [ ]:
import time
import os
import gc
import torch
from pathlib import Path
from huggingface_hub import scan_cache_dir, snapshot_download

print("=" * 80)
print("下载模型权重")
print("=" * 80)

# 配置模型缓存目录
MODEL_CACHE_DIR = WORK_DIR / "huggingface_cache"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(MODEL_CACHE_DIR)

print(f"\n模型缓存目录：{MODEL_CACHE_DIR}")

print("\n检查 Hugging Face 认证...")
try:
    from huggingface_hub import HfApi
    api = HfApi()
    try:
        user_info = api.whoami()
        print(f"  ✓ 已认证：{user_info.get('name', 'Unknown')}")
    except Exception:
        print("  ℹ 未认证（使用匿名访问，私有/受限模型可能无法下载）")
except ImportError:
    print("  ⚠ huggingface_hub 未安装")

print("\n" + "=" * 80)
print("下载 Stable Diffusion 3.5 模型")
print("=" * 80)

MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"
print(f"目标模型：{MODEL_ID}")

def check_model_cached(model_id):
    """
    功能：检查模型是否已在缓存中。

    Check if model is already cached locally.

    Args:
        model_id: Model identifier.

    Returns:
        Boolean indicating if model is cached.
    """
    try:
        cache_info = scan_cache_dir()
        for repo in cache_info.repos:
            if model_id in repo.repo_id:
                return True
        return False
    except Exception:
        return False

if check_model_cached(MODEL_ID):
    print(f"✓ 模型已缓存：{MODEL_ID}")
    print("  跳过下载")
else:
    print(f"\n⏳ 模型未缓存，开始下载：{MODEL_ID}")
    print("   这可能需要 5-20 分钟（取决于网络与镜像）")

    try:
        snapshot_path = snapshot_download(
            repo_id=MODEL_ID,
            cache_dir=str(MODEL_CACHE_DIR),
            local_files_only=False,
        )
        print("  ✓ 模型下载完成")
        print(f"  本地快照路径：{snapshot_path}")
    except Exception as e:
        error_msg = str(e)
        print(f"  ✗ 模型下载失败：{e}")

        if "404" in error_msg or "Entry Not Found" in error_msg:
            print("\n  ❌ 错误：无法访问模型（404）")
            print("  原因：SD3.5 可能需要先在 Hugging Face 页面授权")
        elif "401" in error_msg or "Unauthorized" in error_msg or "403" in error_msg:
            print("\n  ❌ 错误：认证失败（401/403）")
            print("  原因：HF_TOKEN 无效、缺失或未获模型访问权限")
        else:
            print("\n  解决方案：")
            print("    1. 检查网络连接")
            print("    2. 检查 HF_TOKEN 与模型访问授权")
            print("    3. 重新执行本 cell")

        print("\n  ⚠️ 继续执行（后续步骤可能失败）...")

print("\n清理内存...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("  ✓ GPU 缓存已清理")

print("✓ 模型准备完成")

In [ ]:
import os
import shutil
import urllib.request
from pathlib import Path

print("=" * 80)
print("下载 InSPyReNet 权重（paper_full_cuda）")
print("=" * 80)

if 'WORK_DIR' not in globals():
    raise RuntimeError("请先执行前序单元，确保 WORK_DIR 已初始化")

if 'CEG_WM_ROOT' in globals():
    repo_inspyrenet_model_path = Path(CEG_WM_ROOT) / "models" / "inspyrenet" / "ckpt_base.pth"
else:
    repo_inspyrenet_model_path = None

# Colab 同步仓库时不包含未跟踪的 models 目录，因此把可复用权重放在 WORK_DIR 下的持久路径。
cached_inspyrenet_model_path = Path(WORK_DIR) / "models" / "inspyrenet" / "ckpt_base.pth"
inspyrenet_model_path = repo_inspyrenet_model_path or cached_inspyrenet_model_path

cached_inspyrenet_model_path.parent.mkdir(parents=True, exist_ok=True)
if repo_inspyrenet_model_path is not None:
    repo_inspyrenet_model_path.parent.mkdir(parents=True, exist_ok=True)

inspyrenet_repo_tree_url = "https://huggingface.co/plemeri/InSPyReNet/tree/main"
default_inspyrenet_url = "https://huggingface.co/plemeri/InSPyReNet/resolve/main/ckpt_base.pth"

inspyrenet_model_url = os.environ.get(
    "INSPYRENET_MODEL_URL",
    default_inspyrenet_url,
).strip()

force_inspyrenet_download = os.environ.get(
    "INSPYRENET_FORCE_DOWNLOAD",
    "",
).strip().lower() in {"1", "true", "yes"}


def _is_valid_weight_file(file_path) -> bool:
    if file_path is None:
        return False
    return file_path.exists() and file_path.is_file() and file_path.stat().st_size > 0


def _sync_weight_to_repo(source_path: Path, target_path) -> None:
    if target_path is None:
        return
    if source_path.resolve() == target_path.resolve():
        return
    shutil.copy2(source_path, target_path)


print(f"workflow 目标权重路径：{inspyrenet_model_path}")
print(f"持久缓存路径：{cached_inspyrenet_model_path}")
print(f"仓库页面（核对用）：{inspyrenet_repo_tree_url}")
print(f"下载地址（程序使用）：{inspyrenet_model_url}")
print(f"强制重下：{force_inspyrenet_download}")

existing_weight_path = None
candidate_paths = []
if repo_inspyrenet_model_path is not None:
    candidate_paths.append(repo_inspyrenet_model_path)
candidate_paths.append(cached_inspyrenet_model_path)

for candidate_path in candidate_paths:
    if _is_valid_weight_file(candidate_path):
        existing_weight_path = candidate_path
        break

download_required = force_inspyrenet_download or existing_weight_path is None
if existing_weight_path is not None and not force_inspyrenet_download:
    existing_size = existing_weight_path.stat().st_size
    print(f"✓ 已发现现有 InSPyReNet 权重：{existing_weight_path}（size={existing_size} bytes）")
    _sync_weight_to_repo(existing_weight_path, repo_inspyrenet_model_path)
    if existing_weight_path != cached_inspyrenet_model_path:
        shutil.copy2(existing_weight_path, cached_inspyrenet_model_path)
    print("✓ 已复用现有权重，不执行重复下载")

if download_required:
    temp_download_path = cached_inspyrenet_model_path.with_suffix(cached_inspyrenet_model_path.suffix + ".downloading")
    if temp_download_path.exists():
        temp_download_path.unlink()

    print(f"开始下载：{inspyrenet_model_url}")
    urllib.request.urlretrieve(inspyrenet_model_url, str(temp_download_path))

    if not _is_valid_weight_file(temp_download_path):
        temp_download_path.unlink(missing_ok=True)
        raise RuntimeError(f"下载完成但权重文件无效：{temp_download_path}")

    temp_download_path.replace(cached_inspyrenet_model_path)
    _sync_weight_to_repo(cached_inspyrenet_model_path, repo_inspyrenet_model_path)
    print("✓ 下载完成")

if _is_valid_weight_file(repo_inspyrenet_model_path):
    final_weight_path = repo_inspyrenet_model_path
else:
    final_weight_path = cached_inspyrenet_model_path

final_weight_size = final_weight_path.stat().st_size if _is_valid_weight_file(final_weight_path) else 0

print(f"最终 workflow 使用路径：{final_weight_path}")
print(f"最终权重大小：{final_weight_size} bytes")
print("✓ InSPyReNet 准备完成")

## 第 4 步：配置选择、数据准备与 attestation 环境变量

选择工作流配置并准备输入数据，然后设置 formal GPU 验收必需的 attestation 密钥环境变量。

In [ ]:
import json

print("=" * 80)
print("工作流配置和数据准备")
print("=" * 80)

# 选择配置文件。
# 注意：此处 CONFIG_CHOICE 固定为 "paper_full_cuda"，与第 5 步主流程完全对齐。
# 第 5 步不依赖此变量选择 profile，但会检查它是否已定义以确认前序 cell 已执行。
CONFIG_CHOICE = "paper_full_cuda"
print(f"\n选定配置：{CONFIG_CHOICE}")

CONFIG_FILE = CEG_WM_ROOT / "configs" / f"{CONFIG_CHOICE}.yaml"
if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"paper_full_cuda 配置不存在：{CONFIG_FILE}")

PAPER_SPEC_FILE = CEG_WM_ROOT / "configs" / "paper_faithfulness_spec.yaml"
if PAPER_SPEC_FILE.exists():
    print(f"  ✓ 论文一致性规范存在：{PAPER_SPEC_FILE.name}")
else:
    print(f"  ⚠ 未找到论文一致性规范：{PAPER_SPEC_FILE}")

MODEL_CFG_FILE = CEG_WM_ROOT / "configs" / "model_sd3.yaml"
if MODEL_CFG_FILE.exists():
    print(f"  ✓ SD3 模型配置存在：{MODEL_CFG_FILE.name}")
else:
    print(f"  ⚠ 未找到 SD3 模型配置：{MODEL_CFG_FILE}")

print(f"  配置文件路径：{CONFIG_FILE}")
print(f"  配置文件大小：{CONFIG_FILE.stat().st_size / 1024:.1f} KB")

# 创建数据目录
data_dir = CEG_WM_ROOT / "data"
data_dir.mkdir(parents=True, exist_ok=True)

# 准备输入数据
print("\n准备输入数据...")


In [ ]:
import json
import os
import re
import secrets
from datetime import datetime
from pathlib import Path

print("=" * 80)
print("准备 attestation 环境变量（formal GPU 验收必需）")
print("=" * 80)

ATTESTATION_ENV_SPECS = {
    "CEG_WM_K_MASTER": 64,
    "CEG_WM_K_PROMPT": 32,
    "CEG_WM_K_SEED": 32,
}

# 如需固定密钥，请把对应值填成十六进制字符串；留空则优先复用现有环境变量。
USER_ATTESTATION_VALUES = {
    "CEG_WM_K_MASTER": "",
    "CEG_WM_K_PROMPT": "",
    "CEG_WM_K_SEED": "",
}

# 为了让 Colab 首次执行可直接通过 preflight，默认会为缺失项生成本次会话内有效的临时密钥。
AUTO_GENERATE_MISSING_ATTESTATION_KEYS = True

ATTESTATION_INFO_PATH = WORK_DIR / "attestation_env_keys.json"

def _is_valid_hex_secret(value: str, expected_length: int) -> bool:
    if not isinstance(value, str):
        return False
    candidate = value.strip().lower()
    if len(candidate) != expected_length:
        return False
    return re.fullmatch(r"[0-9a-f]+", candidate) is not None

def _mask_secret(value: str) -> str:
    if len(value) <= 8:
        return "*" * len(value)
    return f"{value[:4]}...{value[-4:]}"

resolved_values = {}
generated_env_vars = []
invalid_env_vars = []

for env_name, expected_length in ATTESTATION_ENV_SPECS.items():
    manual_value = USER_ATTESTATION_VALUES.get(env_name, "")
    existing_value = os.environ.get(env_name, "")

    if _is_valid_hex_secret(manual_value, expected_length):
        resolved_value = manual_value.strip().lower()
        source = "manual_input"
    elif _is_valid_hex_secret(existing_value, expected_length):
        resolved_value = existing_value.strip().lower()
        source = "existing_env"
    elif AUTO_GENERATE_MISSING_ATTESTATION_KEYS:
        resolved_value = secrets.token_hex(expected_length // 2)
        source = "generated_for_session"
        generated_env_vars.append(env_name)
    else:
        resolved_value = ""
        source = "missing"

    if resolved_value:
        os.environ[env_name] = resolved_value
        resolved_values[env_name] = {
            "length": len(resolved_value),
            "masked_value": _mask_secret(resolved_value),
            "value": resolved_value,
            "source": source,
        }
    else:
        invalid_env_vars.append(env_name)

print("\nattestation 环境变量状态：")
for env_name in ATTESTATION_ENV_SPECS:
    if env_name in resolved_values:
        item = resolved_values[env_name]
        print(
            f"  ✓ {env_name}: source={item['source']}, len={item['length']}, value={item['masked_value']}"
        )
    else:
        print(f"  ✗ {env_name}: 缺失或格式非法")

if generated_env_vars:
    print("\nℹ 已为当前 Notebook 会话自动生成临时 attestation 密钥：")
    for env_name in generated_env_vars:
        print(f"  - {env_name}")
    print("  重新启动内核后需要再次执行本 cell。")

if invalid_env_vars:
    raise RuntimeError(
        "以下 attestation 环境变量仍未就绪，请填写合法十六进制值或保持自动生成开启："
        f" {invalid_env_vars}"
    )

attestation_info = {
    "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "work_dir": str(WORK_DIR),
    "auto_generate_missing_attestation_keys": bool(AUTO_GENERATE_MISSING_ATTESTATION_KEYS),
    "generated_env_vars": generated_env_vars,
    "env_specs": ATTESTATION_ENV_SPECS,
    "resolved_values": resolved_values,
}

with open(ATTESTATION_INFO_PATH, "w", encoding="utf-8") as f:
    json.dump(attestation_info, f, indent=2, ensure_ascii=False)

print(f"\n✓ attestation 环境变量与密钥已保存：{ATTESTATION_INFO_PATH}")
print("✓ attestation 环境变量准备完成，可继续执行运行前预检与第 5 步")

In [ ]:
import json
import os
import shutil
import socket
from pathlib import Path

import torch

print("=" * 80)
print("第 4.9 步：output-only workflow 预检")
print("=" * 80)

if 'CEG_WM_ROOT' not in globals():
    raise RuntimeError("请先执行前面的环境初始化 cell，确保 CEG_WM_ROOT 已定义")

precheck_results = []

def _record_check(name: str, ok: bool, detail: str) -> None:
    status = "✓" if ok else "✗"
    print(f"{status} {name}: {detail}")
    precheck_results.append({"name": name, "ok": bool(ok), "detail": detail})

python_ok = os.environ.get("VIRTUAL_ENV") or os.environ.get("CONDA_DEFAULT_ENV") or sys.executable
_record_check("Python 运行时", True, str(python_ok))

try:
    cuda_ok = torch.cuda.is_available()
    if cuda_ok:
        gpu_name = torch.cuda.get_device_name(0)
        _record_check("CUDA 可用", True, f"device={gpu_name}")
    else:
        _record_check("CUDA 可用", False, "未检测到 GPU，请在 Colab 切换到 GPU Runtime")
except Exception as exc:
    _record_check("CUDA 可用", False, f"torch 检查异常: {type(exc).__name__}: {exc}")

required_paths = [
    CEG_WM_ROOT / "configs" / "paper_full_cuda.yaml",
    CEG_WM_ROOT / "scripts" / "run_paper_full_cuda.py",
    CEG_WM_ROOT / "scripts" / "run_experiment_matrix.py",
]
for path in required_paths:
    _record_check(f"文件存在: {path.name}", path.exists(), str(path))

attestation_env_specs = {
    "CEG_WM_K_MASTER": 64,
    "CEG_WM_K_PROMPT": 32,
    "CEG_WM_K_SEED": 32,
}
for env_name, expected_length in attestation_env_specs.items():
    value = os.environ.get(env_name, "").strip().lower()
    is_hex = len(value) == expected_length and all(ch in "0123456789abcdef" for ch in value)
    detail = f"len={len(value)}" if value else "<absent>"
    _record_check(f"attestation 环境变量: {env_name}", is_hex, detail)

inspyrenet_weight_path = CEG_WM_ROOT / "models" / "inspyrenet" / "ckpt_base.pth"
if inspyrenet_weight_path.exists() and inspyrenet_weight_path.is_file():
    try:
        weight_size = inspyrenet_weight_path.stat().st_size
        inspyrenet_ok = weight_size > 0
        _record_check(
            "InSPyReNet 权重存在",
            inspyrenet_ok,
            f"path={inspyrenet_weight_path}, size={weight_size} bytes",
        )
    except Exception as exc:
        _record_check(
            "InSPyReNet 权重存在",
            False,
            f"读取文件信息失败: {type(exc).__name__}: {exc}",
        )
else:
    _record_check(
        "InSPyReNet 权重存在",
        False,
        f"未找到权重文件: {inspyrenet_weight_path}（请先执行第 3 步 InSPyReNet 下载单元）",
    )

required_modules = ["yaml", "huggingface_hub", "diffusers", "transformers", "safetensors"]
for module_name in required_modules:
    try:
        __import__(module_name)
        _record_check(f"模块可导入: {module_name}", True, "ok")
    except Exception as exc:
        _record_check(f"模块可导入: {module_name}", False, f"{type(exc).__name__}: {exc}")

hf_ok = False
hf_detail = "未执行"
try:
    from huggingface_hub import HfApi

    socket.setdefaulttimeout(15)
    api = HfApi()
    _ = api.model_info("stabilityai/stable-diffusion-3.5-medium")
    hf_ok = True
    hf_detail = "可访问 stabilityai/stable-diffusion-3.5-medium"
except Exception as exc:
    hf_ok = False
    hf_detail = f"访问失败（可能是网络/权限问题）: {type(exc).__name__}: {exc}"
_record_check("HF 模型可访问", hf_ok, hf_detail)

usage = shutil.disk_usage(str(CEG_WM_ROOT))
free_gb = usage.free / (1024 ** 3)
disk_ok = free_gb >= 20.0
_record_check("磁盘剩余空间", disk_ok, f"free={free_gb:.1f} GB，建议 >= 20 GB")

hard_fail_names = {
    "CUDA 可用",
    "文件存在: paper_full_cuda.yaml",
    "文件存在: run_paper_full_cuda.py",
    "InSPyReNet 权重存在",
    "模块可导入: diffusers",
    "模块可导入: transformers",
}
hard_fail = [item for item in precheck_results if (item["name"] in hard_fail_names and not item["ok"])]
soft_fail = [item for item in precheck_results if (item["name"] not in hard_fail_names and not item["ok"])]

print("\n" + "-" * 80)
print("预检结果汇总")
print("-" * 80)
print(f"硬性失败项数量: {len(hard_fail)}")
print(f"软性失败项数量: {len(soft_fail)}")

if hard_fail:
    print("\n✗ 预检未通过（存在硬性失败项），请先修复后再执行第 5 步。")
    for item in hard_fail:
        print(f"  - {item['name']}: {item['detail']}")
else:
    if soft_fail:
        print("\n✓ 预检通过（存在软性风险项，可继续执行 output-only workflow）：")
        for item in soft_fail:
            print(f"  - {item['name']}: {item['detail']}")
    else:
        print("\n✓ 预检全部通过，可以执行第 5 步。")

## 第 5 步：运行 paper_full_cuda 项目输出 workflow

本步骤调用 scripts/run_paper_full_cuda.py，直接使用 configs/paper_full_cuda.yaml 生成项目最终输出。

执行目标：
- 运行 embed、detect、calibrate、evaluate 主链。
- 如配置启用 experiment_matrix，则补充生成矩阵输出。
- 不执行 formal acceptance。
- 不执行 signoff。

执行后分支：
- 如果本步完成，下一步先执行第 5.1 步，优先检查并导出完整项目结果包。
- 如果第 5.1 步提示缺失必需产物并禁止打包，则继续执行第 6 步验证输出，再执行第 7 步生成失败诊断包。

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

import torch

print("=" * 80)
print("第 5 步：paper_full_cuda 项目输出 workflow")
print("=" * 80)

if 'CEG_WM_ROOT' not in globals() or 'CONFIG_CHOICE' not in globals():
    raise RuntimeError("请先执行前面的环境与配置 cell（第 1-4 步）")

if not torch.cuda.is_available():
    raise RuntimeError("当前运行时未检测到 CUDA。请在 Colab 切换到 GPU Runtime 后重试。")

gpu_name = torch.cuda.get_device_name(0)
print(f"✓ CUDA 可用，设备：{gpu_name}")

RUN_ROOT = CEG_WM_ROOT / "outputs" / "colab_run_paper_full_cuda"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

for name in ["records", "artifacts", "logs", "outputs"]:
    target = RUN_ROOT / name
    if target.exists():
        shutil.rmtree(target)

logs_dir = RUN_ROOT / "logs"
logs_dir.mkdir(parents=True, exist_ok=True)

ACTIVE_CONFIG_FILE = CEG_WM_ROOT / "configs" / "paper_full_cuda.yaml"
ACTIVE_ENTRYPOINT = CEG_WM_ROOT / "scripts" / "run_paper_full_cuda.py"
if not ACTIVE_CONFIG_FILE.exists():
    raise FileNotFoundError(f"未找到项目配置：{ACTIVE_CONFIG_FILE}")
if not ACTIVE_ENTRYPOINT.exists():
    raise FileNotFoundError(f"未找到项目脚本：{ACTIVE_ENTRYPOINT}")

ACTIVE_WORKFLOW_PROFILE = "paper_full_cuda_outputs"
ACTIVE_SIGNOFF_PROFILE = None
EXPECTED_PRIMARY_OUTPUTS = [
    RUN_ROOT / "records" / "embed_record.json",
    RUN_ROOT / "records" / "detect_record.json",
    RUN_ROOT / "records" / "calibration_record.json",
    RUN_ROOT / "records" / "evaluate_record.json",
    RUN_ROOT / "artifacts" / "thresholds" / "thresholds_artifact.json",
    RUN_ROOT / "artifacts" / "evaluation_report.json",
]

print(f"✓ 输出目录：{RUN_ROOT}")
print(f"✓ 运行配置：{ACTIVE_CONFIG_FILE}")
print(f"✓ 主入口脚本：{ACTIVE_ENTRYPOINT}")

command = [
    sys.executable,
    str(ACTIVE_ENTRYPOINT),
    "--config",
    str(ACTIVE_CONFIG_FILE),
    "--run-root",
    str(RUN_ROOT),
]

print("\n执行 paper_full_cuda 项目输出脚本...")
result = subprocess.run(
    command,
    cwd=str(CEG_WM_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

workflow_log = logs_dir / "workflow_execution.log"
workflow_stdout_full_log = logs_dir / "workflow_stdout_full.log"
workflow_stderr_full_log = logs_dir / "workflow_stderr_full.log"
workflow_command_json = logs_dir / "workflow_command.json"

with open(workflow_stdout_full_log, "w", encoding="utf-8") as f:
    f.write(result.stdout or "")

with open(workflow_stderr_full_log, "w", encoding="utf-8") as f:
    f.write(result.stderr or "")

command_meta = {
    "command": command,
    "cwd": str(CEG_WM_ROOT),
    "return_code": int(result.returncode),
    "stdout_log": str(workflow_stdout_full_log),
    "stderr_log": str(workflow_stderr_full_log),
    "mode": "project_outputs_only",
}
with open(workflow_command_json, "w", encoding="utf-8") as f:
    json.dump(command_meta, f, indent=2, ensure_ascii=False)

with open(workflow_log, "w", encoding="utf-8") as f:
    f.write("COMMAND:\n")
    f.write(" ".join(command))
    f.write("\n\nRETURN_CODE:\n")
    f.write(str(result.returncode))
    f.write("\n\nSTDOUT:\n")
    f.write(result.stdout or "")
    f.write("\n\nSTDERR:\n")
    f.write(result.stderr or "")

PROJECT_OUTPUT_SUMMARY = {
    "mode": "project_outputs_only",
    "entrypoint": str(ACTIVE_ENTRYPOINT),
    "run_root": str(RUN_ROOT),
    "required_outputs": [str(path.relative_to(RUN_ROOT)) for path in EXPECTED_PRIMARY_OUTPUTS],
    "required_outputs_present": {
        str(path.relative_to(RUN_ROOT)): path.exists() for path in EXPECTED_PRIMARY_OUTPUTS
    },
}

print(f"返回码：{result.returncode}")
print(f"完整 stdout 日志：{workflow_stdout_full_log}")
print(f"完整 stderr 日志：{workflow_stderr_full_log}")
print(f"命令元信息：{workflow_command_json}")

if result.stdout:
    print("\nSTDOUT（最后 30 行）：")
    print("\n".join(result.stdout.splitlines()[-30:]))

if result.stderr:
    print("\nSTDERR（最后 20 行）：")
    print("\n".join(result.stderr.splitlines()[-20:]))

if result.returncode == 0:
    STAGE_STATUS = "success"
else:
    STAGE_STATUS = f"failed (rc={result.returncode})"

print("\n" + "=" * 80)
if result.returncode == 0:
    print("✓ 第 5 步执行成功！")
    print(f"  结果目录：{RUN_ROOT}")
    print("  当前路径仅以项目最终输出为目标，不检查 formal acceptance / signoff。")
else:
    print("✗ 第 5 步执行失败")
    print(f"  返回码：{result.returncode}")
    print(f"  日志文件：{workflow_log}")
    print("\n提示：请运行下一个 cell（第 6 步验证或第 7 步诊断）查看具体问题")

print("=" * 80)

In [ ]:
import json
import shutil
import subprocess
import zipfile
from datetime import datetime
from pathlib import Path

import yaml

print("=" * 80)
print("优先打包项目最终输出")
print("=" * 80)

if 'RUN_ROOT' not in globals():
    raise RuntimeError("请先执行第 5 步，生成 RUN_ROOT 后再打包项目结果")

run_root = Path(RUN_ROOT)
records_dir = run_root / "records"
artifacts_dir = run_root / "artifacts"
outputs_dir = run_root / "outputs"
logs_dir = run_root / "logs"

PROJECT_EXPORT_STATUS = "not_started"
PROJECT_EXPORT_DOWNLOADED = False
PROJECT_RESULTS_ARCHIVE_PATH = None
PROJECT_RESULTS_ARCHIVE_SYNC_PATH = None
PROJECT_EXPORT_MISSING_FILES = []
PROJECT_EXPORT_OPTIONAL_FILES = []
PROJECT_EXPORT_FILE_COUNT = 0

def _ensure_git_info_file() -> Path:
    existing_git_info_path = globals().get("GIT_INFO_PATH")
    if existing_git_info_path is not None:
        candidate_path = Path(existing_git_info_path)
        if candidate_path.exists() and candidate_path.is_file():
            return candidate_path

    if 'CEG_WM_ROOT' not in globals():
        raise RuntimeError("缺少 CEG_WM_ROOT，无法补生成 Git 信息")

    repo_root = Path(CEG_WM_ROOT)
    git_info_path = WORK_DIR / "git_version_info.json"

    def _run_git(command):
        result = subprocess.run(
            command,
            cwd=str(repo_root),
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
        )
        if result.returncode != 0:
            raise RuntimeError(
                f"Git 信息补生成失败，return_code={result.returncode}，command={' '.join(command)}"
            )
        return result.stdout.strip()

    version_info = {
        "repo_url": _run_git(["git", "remote", "get-url", "origin"]),
        "branch": _run_git(["git", "rev-parse", "--abbrev-ref", "HEAD"]),
        "target_dir": str(repo_root),
        "sync_mode": globals().get("sync_mode", "<unknown>"),
        "outputs_cleaned": True,
        "commit_hash": _run_git(["git", "rev-parse", "HEAD"]),
        "commit_hash_short": _run_git(["git", "rev-parse", "--short=8", "HEAD"]),
        "commit_message": _run_git(["git", "log", "-1", "--pretty=%s"]),
        "commit_timestamp": _run_git(["git", "log", "-1", "--pretty=%ci"]),
        "commit_author": _run_git(["git", "log", "-1", "--pretty=%an <%ae>"]),
        "sync_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "generated_for_export": True,
    }

    with open(git_info_path, "w", encoding="utf-8") as f:
        json.dump(version_info, f, indent=2, ensure_ascii=False)

    globals()["GIT_INFO_PATH"] = git_info_path
    print(f"✓ 已补生成 Git 信息：{git_info_path}")
    return git_info_path

def _build_timestamped_filename(base_name: str, suffix: str) -> str:
    timestamp_suffix = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{base_name}_{timestamp_suffix}{suffix}"

def _sync_file_to_google_drive(source_path: Path) -> Path | None:
    drive_export_dir = globals().get("GOOGLE_DRIVE_EXPORT_DIR")
    if drive_export_dir is None:
        return None

    drive_export_dir = Path(drive_export_dir)
    drive_export_dir.mkdir(parents=True, exist_ok=True)
    target_path = drive_export_dir / source_path.name
    shutil.copy2(source_path, target_path)
    return target_path

def _load_runtime_config() -> dict:
    config_path = Path(globals().get("ACTIVE_CONFIG_FILE", ""))
    if not config_path.exists():
        return {}
    payload = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    return payload if isinstance(payload, dict) else {}

try:
    git_info_path = _ensure_git_info_file()
except Exception as exc:
    git_info_path = None
    PROJECT_EXPORT_MISSING_FILES.append(f"<Git 信息不可用: {exc}>")

runtime_cfg = _load_runtime_config()
matrix_enabled = isinstance(runtime_cfg.get("experiment_matrix"), dict)

required_files = [
    records_dir / "embed_record.json",
    records_dir / "detect_record.json",
    records_dir / "calibration_record.json",
    records_dir / "evaluate_record.json",
    artifacts_dir / "evaluation_report.json",
    artifacts_dir / "run_closure.json",
    artifacts_dir / "thresholds" / "thresholds_artifact.json",
]
required_files = [Path(path) for path in required_files]

optional_files = []
if matrix_enabled:
    optional_files.append(outputs_dir / "experiment_matrix" / "artifacts" / "grid_summary.json")

required_support_files = [
    logs_dir / "workflow_execution.log",
    logs_dir / "workflow_stdout_full.log",
    logs_dir / "workflow_stderr_full.log",
    logs_dir / "workflow_command.json",
]
if git_info_path is not None:
    required_support_files.append(git_info_path)

missing_required_files = [str(path) for path in required_files if not path.exists()]
missing_support_files = [str(path) for path in required_support_files if not Path(path).exists()]
missing_optional_files = [str(path) for path in optional_files if not Path(path).exists()]
PROJECT_EXPORT_MISSING_FILES.extend(missing_required_files)
PROJECT_EXPORT_MISSING_FILES.extend(missing_support_files)
PROJECT_EXPORT_OPTIONAL_FILES.extend(missing_optional_files)

if PROJECT_EXPORT_MISSING_FILES:
    PROJECT_EXPORT_STATUS = "missing_required_files"
    print("✗ 检测到缺失必需产物，当前禁止打包：")
    for item in PROJECT_EXPORT_MISSING_FILES:
        print(f"  - {item}")
    print("请先执行第 6 步验证工作流输出，再执行第 7 步快速诊断导出。")
else:
    package_manifest = {
        "run_root": str(run_root),
        "profile": globals().get("ACTIVE_WORKFLOW_PROFILE", "paper_full_cuda_outputs"),
        "stage_status": globals().get("STAGE_STATUS", "<absent>"),
        "required_files": [str(path.relative_to(run_root)) for path in required_files],
        "optional_files": [
            str(path.relative_to(run_root)) for path in optional_files if path.exists()
        ],
        "missing_optional_files": PROJECT_EXPORT_OPTIONAL_FILES,
        "required_support_files": [
            str(path.relative_to(run_root)) if str(path).startswith(str(run_root)) else path.name
            for path in required_support_files
        ],
        "git_info_file": git_info_path.name if git_info_path is not None else "<absent>",
        "mode": "project_outputs_only",
    }
    manifest_path = artifacts_dir / "package_manifest.json"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(package_manifest, f, indent=2, ensure_ascii=False)
    print(f"✓ 已写入打包清单：{manifest_path}")
    if PROJECT_EXPORT_OPTIONAL_FILES:
        print("ℹ 以下可选输出缺失，但不阻断项目结果打包：")
        for item in PROJECT_EXPORT_OPTIONAL_FILES:
            print(f"  - {item}")

    def _iter_project_export_files():
        scoped_dirs = [
            (records_dir, "records"),
            (artifacts_dir, "artifacts"),
            (outputs_dir, "outputs"),
            (logs_dir, "logs"),
        ]
        for base_dir, prefix in scoped_dirs:
            if not base_dir.exists():
                continue
            for file_path in sorted(base_dir.rglob("*")):
                if not file_path.is_file():
                    continue
                yield file_path, f"{prefix}/{file_path.relative_to(base_dir).as_posix()}"

    archive_name = _build_timestamped_filename(f"{CONFIG_CHOICE}_Project_Outputs", ".zip")
    archive_path = WORK_DIR / archive_name
    seen_arcnames = set()

    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for file_path, arcname in _iter_project_export_files():
            if arcname in seen_arcnames:
                continue
            seen_arcnames.add(arcname)
            zf.write(file_path, arcname=arcname)
        if git_info_path is not None and git_info_path.exists():
            arcname = f"runtime_metadata/{git_info_path.name}"
            if arcname not in seen_arcnames:
                seen_arcnames.add(arcname)
                zf.write(git_info_path, arcname=arcname)

    PROJECT_EXPORT_STATUS = "archive_created"
    PROJECT_RESULTS_ARCHIVE_PATH = archive_path
    PROJECT_EXPORT_FILE_COUNT = len(seen_arcnames)
    PROJECT_RESULTS_ARCHIVE_SYNC_PATH = _sync_file_to_google_drive(archive_path)

    print(f"✓ 完整项目结果包：{archive_path}")
    print(f"  文件数量：{PROJECT_EXPORT_FILE_COUNT}")
    if PROJECT_RESULTS_ARCHIVE_SYNC_PATH is not None:
        print(f"✓ 已同步到 Google Drive：{PROJECT_RESULTS_ARCHIVE_SYNC_PATH}")
    else:
        print("ℹ 未检测到 Google Drive 同步目录，跳过同步")

    if 'IN_COLAB' in globals() and IN_COLAB:
        try:
            from google.colab import files as colab_files

            colab_files.download(str(archive_path))
            PROJECT_EXPORT_STATUS = "download_triggered"
            PROJECT_EXPORT_DOWNLOADED = True
            print(f"✓ 已触发下载：{archive_path.name}")
        except Exception as exc:
            print(f"⚠ 自动下载失败，请手动下载：{archive_path}，原因：{exc}")

## 第 6 步：验证工作流输出

检查工作流是否成功生成了所有必要的输出文件。

In [ ]:
import json
from pathlib import Path

import yaml

print("=" * 80)
print("验证项目输出 workflow")
print("=" * 80)

if 'RUN_ROOT' not in globals():
    raise RuntimeError("请先执行第 5 步，生成 RUN_ROOT 后再验证输出")

records_dir = RUN_ROOT / "records"
artifacts_dir = RUN_ROOT / "artifacts"
outputs_dir = RUN_ROOT / "outputs"
config_path = Path(globals().get("ACTIVE_CONFIG_FILE", ""))

def _load_json_dict(path: Path):
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)
    return payload if isinstance(payload, dict) else {}

runtime_cfg = {}
if config_path.exists():
    payload = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    runtime_cfg = payload if isinstance(payload, dict) else {}
matrix_enabled = isinstance(runtime_cfg.get("experiment_matrix"), dict)

print("\n[1/5] 检查记录文件...")
expected_records = [
    "embed_record.json",
    "detect_record.json",
    "calibration_record.json",
    "evaluate_record.json",
]

records_found = {}
for record_name in expected_records:
    record_path = records_dir / record_name
    exists = record_path.exists()
    records_found[record_name] = exists
    print(f"  {'✓' if exists else '✗'} {record_name}")

print("\n[2/5] 检查项目输出工件...")
expected_artifacts = [
    ("evaluation_report.json", artifacts_dir / "evaluation_report.json"),
    ("run_closure.json", artifacts_dir / "run_closure.json"),
    ("thresholds/thresholds_artifact.json", artifacts_dir / "thresholds" / "thresholds_artifact.json"),
]
optional_artifacts = []
if matrix_enabled:
    optional_artifacts.append(
        ("outputs/experiment_matrix/artifacts/grid_summary.json", outputs_dir / "experiment_matrix" / "artifacts" / "grid_summary.json")
    )

artifacts_found = {}
for artifact_name, artifact_path in expected_artifacts:
    artifacts_found[artifact_name] = artifact_path.exists()
    print(f"  {'✓' if artifacts_found[artifact_name] else '✗'} {artifact_name}")

optional_artifacts_found = {}
for artifact_name, artifact_path in optional_artifacts:
    optional_artifacts_found[artifact_name] = artifact_path.exists()
    print(f"  {'✓' if optional_artifacts_found[artifact_name] else 'ℹ'} {artifact_name}（可选）")

print("\n[3/5] 检查 detect 输出语义...")
detect_record = _load_json_dict(records_dir / "detect_record.json")
final_decision = detect_record.get("final_decision") if isinstance(detect_record.get("final_decision"), dict) else {}
detect_threshold_source = detect_record.get("threshold_source") or final_decision.get("threshold_source")
detect_decision_status = final_decision.get("decision_status", "<absent>")
content_payload = detect_record.get("content_evidence_payload") if isinstance(detect_record.get("content_evidence_payload"), dict) else {}
content_score = content_payload.get("score")
print(f"  threshold_source: {detect_threshold_source}")
print(f"  final_decision.decision_status: {detect_decision_status}")
print(f"  content_evidence_payload.score: {content_score}")
observation_or_final_ok = detect_decision_status in {"abstain", "decided"}

print("\n[4/5] 检查 evaluate 输出语义...")
evaluate_record = _load_json_dict(records_dir / "evaluate_record.json")
evaluation_report = _load_json_dict(artifacts_dir / "evaluation_report.json")
metrics_payload = evaluate_record.get("metrics") if isinstance(evaluate_record.get("metrics"), dict) else {}
semantic_ok = bool(evaluation_report) and isinstance(metrics_payload, dict)
print(f"  evaluation_report 存在: {bool(evaluation_report)}")
print(f"  metrics keys: {sorted(metrics_payload.keys())[:8] if metrics_payload else []}")

print("\n[5/5] 汇总结论...")
all_required_records = all(records_found.values())
all_required_artifacts = all(artifacts_found.values())
project_outputs_complete = all_required_records and all_required_artifacts and observation_or_final_ok and semantic_ok

VALIDATION_SUMMARY = {
    "mode": "project_outputs_only",
    "records_found": records_found,
    "artifacts_found": artifacts_found,
    "optional_artifacts_found": optional_artifacts_found,
    "detect_threshold_source": detect_threshold_source,
    "detect_decision_status": detect_decision_status,
    "observation_or_final_ok": observation_or_final_ok,
    "semantic_ok": semantic_ok,
    "project_outputs_complete": project_outputs_complete,
    "matrix_enabled": matrix_enabled,
}

print(f"  records 完整：{all_required_records}")
print(f"  artifacts 完整：{all_required_artifacts}")
if optional_artifacts_found:
    print(f"  可选 matrix 输出存在：{all(optional_artifacts_found.values())}")
print(f"  detect 语义合格：{observation_or_final_ok}")
print(f"  evaluation_report 语义合格：{semantic_ok}")
print(f"  项目输出完整：{project_outputs_complete}")
print(f"  输出目录：{RUN_ROOT}")

## 第 7 步：失败时快速诊断与审计包导出

如果第 5.1 步检测到缺失必需产物而禁止打包，请先完成第 6 步验证，再执行本步生成诊断与审计包。

如果第 5.1 步已经成功导出完整项目结果，本步只做快速诊断，不再重复触发下载。

In [ ]:
import json
import shutil
import subprocess
import zipfile
from datetime import datetime
from pathlib import Path

print("=" * 80)
print("第 7 步：快速诊断与按需导出")
print("=" * 80)

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

if 'RUN_ROOT' not in globals():
    raise RuntimeError("请先执行第 5 步，生成 RUN_ROOT 后再执行快速诊断")

run_root = Path(RUN_ROOT)
if not run_root.exists():
    raise RuntimeError(f"RUN_ROOT 不存在：{run_root}")

project_export_success = globals().get("PROJECT_EXPORT_STATUS") in {"archive_created", "download_triggered"}
validation_summary = globals().get("VALIDATION_SUMMARY", {})

def _load_json(path: Path):
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def _ensure_git_info_file() -> Path | None:
    if 'GIT_INFO_PATH' in globals():
        candidate_path = Path(GIT_INFO_PATH)
        if candidate_path.exists() and candidate_path.is_file():
            return candidate_path

    if 'CEG_WM_ROOT' not in globals():
        return None

    repo_root = Path(CEG_WM_ROOT)
    git_info_path = WORK_DIR / "git_version_info.json"

    def _run_git(command):
        result = subprocess.run(
            command,
            cwd=str(repo_root),
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
        )
        if result.returncode != 0:
            raise RuntimeError(
                f"Git 信息补生成失败，return_code={result.returncode}，command={' '.join(command)}"
            )
        return result.stdout.strip()

    version_info = {
        "repo_url": _run_git(["git", "remote", "get-url", "origin"]),
        "branch": _run_git(["git", "rev-parse", "--abbrev-ref", "HEAD"]),
        "target_dir": str(repo_root),
        "sync_mode": globals().get("sync_mode", "<unknown>"),
        "outputs_cleaned": True,
        "commit_hash": _run_git(["git", "rev-parse", "HEAD"]),
        "commit_hash_short": _run_git(["git", "rev-parse", "--short=8", "HEAD"]),
        "commit_message": _run_git(["git", "log", "-1", "--pretty=%s"]),
        "commit_timestamp": _run_git(["git", "log", "-1", "--pretty=%ci"]),
        "commit_author": _run_git(["git", "log", "-1", "--pretty=%an <%ae>"]),
        "sync_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "generated_for_export": True,
    }

    with open(git_info_path, "w", encoding="utf-8") as f:
        json.dump(version_info, f, indent=2, ensure_ascii=False)

    globals()["GIT_INFO_PATH"] = git_info_path
    print(f"✓ 已补生成 Git 信息：{git_info_path}")
    return git_info_path

def _build_timestamped_filename(base_name: str, suffix: str) -> str:
    timestamp_suffix = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{base_name}_{timestamp_suffix}{suffix}"

def _sync_file_to_google_drive(source_path: Path) -> Path | None:
    drive_export_dir = globals().get("GOOGLE_DRIVE_EXPORT_DIR")
    if drive_export_dir is None:
        return None

    drive_export_dir = Path(drive_export_dir)
    drive_export_dir.mkdir(parents=True, exist_ok=True)
    target_path = drive_export_dir / source_path.name
    shutil.copy2(source_path, target_path)
    return target_path

git_info_error = None
try:
    git_info_path = _ensure_git_info_file()
except Exception as exc:
    git_info_path = None
    git_info_error = str(exc)

detect_record_path = run_root / "records" / "detect_record.json"
run_closure_path = run_root / "artifacts" / "run_closure.json"
evaluation_report_path = run_root / "artifacts" / "evaluation_report.json"
grid_summary_path = run_root / "outputs" / "experiment_matrix" / "artifacts" / "grid_summary.json"
workflow_log_path = run_root / "logs" / "workflow_execution.log"

detect_record = _load_json(detect_record_path)
run_closure = _load_json(run_closure_path)
evaluation_report = _load_json(evaluation_report_path)
grid_summary = _load_json(grid_summary_path)

failed_step = None
workflow_return_code = None
if workflow_log_path.exists():
    with open(workflow_log_path, "r", encoding="utf-8") as f:
        log_lines = f.read().splitlines()
    for idx, line in enumerate(log_lines):
        if line.strip() == "RETURN_CODE:" and idx + 1 < len(log_lines):
            try:
                workflow_return_code = int(log_lines[idx + 1].strip())
            except Exception:
                workflow_return_code = None

missing_outputs = []
for required_path in [
    run_root / "records" / "embed_record.json",
    run_root / "records" / "detect_record.json",
    run_root / "records" / "calibration_record.json",
    run_root / "records" / "evaluate_record.json",
    run_root / "artifacts" / "thresholds" / "thresholds_artifact.json",
    run_root / "artifacts" / "evaluation_report.json",
    run_root / "artifacts" / "run_closure.json",
]:
    if not required_path.exists():
        missing_outputs.append(str(required_path.relative_to(run_root)))

print(f"项目结果包状态：{globals().get('PROJECT_EXPORT_STATUS', '<absent>')}")
print(f"完整项目结果包：{globals().get('PROJECT_RESULTS_ARCHIVE_PATH', '<absent>')}")
print(f"是否已成功导出完整项目结果：{project_export_success}")
print(f"Git 信息文件：{git_info_path if git_info_path is not None else '<absent>'}")
if git_info_error:
    print(f"Git 信息补生成错误：{git_info_error}")

print("\n[1/4] 运行闭包状态")
if run_closure:
    print(f"  status_ok: {run_closure.get('status_ok')}")
    print(f"  status_reason: {run_closure.get('status_reason')}")
else:
    print("  ✗ run_closure 缺失")

print("\n[2/4] detect / evaluate 输出状态")
final_decision = detect_record.get("final_decision", {}) if isinstance(detect_record, dict) else {}
print(f"  detect threshold_source: {detect_record.get('threshold_source', '<absent>')}")
print(f"  detect final_decision: {final_decision}")
print(f"  evaluation_report 存在: {bool(evaluation_report)}")
print(f"  grid_summary 存在: {bool(grid_summary)}")

print("\n[3/4] workflow / validation 状态")
print(f"  workflow_return_code: {workflow_return_code}")
print(f"  failed_step: {failed_step}")
print(f"  missing_outputs: {missing_outputs}")
print(f"  validation_summary: {validation_summary if validation_summary else '<absent>'}")

print("\n[4/4] 生成诊断包")
diagnosis_lines = [
    "=" * 80,
    "CEG-WM Output Workflow Diagnosis",
    "=" * 80,
    f"generated_at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"run_root: {run_root}",
    f"stage_status: {globals().get('STAGE_STATUS', '<absent>')}",
    f"project_export_status: {globals().get('PROJECT_EXPORT_STATUS', '<absent>')}",
    f"project_results_archive_path: {globals().get('PROJECT_RESULTS_ARCHIVE_PATH', '<absent>')}",
    f"git_info_path: {git_info_path if git_info_path is not None else '<absent>'}",
    f"git_info_error: {git_info_error if git_info_error else '<absent>'}",
    f"workflow_return_code: {workflow_return_code}",
    f"missing_outputs: {missing_outputs}",
    "",
    "[validation_summary]",
    json.dumps(validation_summary, indent=2, ensure_ascii=False) if validation_summary else "<absent>",
    "",
    "[run_closure]",
    json.dumps(run_closure, indent=2, ensure_ascii=False) if run_closure else "<absent>",
    "",
    "[detect_record.final_decision]",
    json.dumps(final_decision, indent=2, ensure_ascii=False) if final_decision else "<absent>",
    "",
    "[evaluation_report_exists]",
    json.dumps({"exists": bool(evaluation_report)}, indent=2, ensure_ascii=False),
]

diagnosis_path = run_root / "logs" / "DIAGNOSIS.txt"
diagnosis_path.parent.mkdir(parents=True, exist_ok=True)
with open(diagnosis_path, "w", encoding="utf-8") as f:
    f.write("\n".join(diagnosis_lines))

zip_name = _build_timestamped_filename(f"{CONFIG_CHOICE}_Diagnostic", ".zip")
zip_path = WORK_DIR / zip_name
seen_arcnames = set()

def _write_unique_file(zf, full_path: Path, arcname: str):
    if arcname in seen_arcnames:
        return
    if not full_path.exists() or not full_path.is_file():
        return
    seen_arcnames.add(arcname)
    zf.write(full_path, arcname=arcname)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for base_dir, prefix in [
        (run_root / "records", "records"),
        (run_root / "artifacts", "artifacts"),
        (run_root / "outputs", "outputs"),
        (run_root / "logs", "logs"),
    ]:
        if not base_dir.exists():
            continue
        for file_path in sorted(base_dir.rglob("*")):
            if not file_path.is_file():
                continue
            relative_path = file_path.relative_to(base_dir).as_posix()
            arcname = f"{prefix}/{relative_path}"
            _write_unique_file(zf, file_path, arcname)

    if git_info_path is not None:
        _write_unique_file(zf, git_info_path, f"runtime_metadata/{git_info_path.name}")

DIAGNOSTIC_ARCHIVE_PATH = zip_path
DIAGNOSTIC_ARCHIVE_SYNC_PATH = _sync_file_to_google_drive(zip_path)
print(f"✓ 诊断包生成：{zip_path}")
print(f"  去重后文件数量：{len(seen_arcnames)}")
if DIAGNOSTIC_ARCHIVE_SYNC_PATH is not None:
    print(f"✓ 已同步到 Google Drive：{DIAGNOSTIC_ARCHIVE_SYNC_PATH}")
else:
    print("ℹ 未检测到 Google Drive 同步目录，跳过同步")

if project_export_success:
    print("ℹ 前序已经成功导出完整项目结果，本步不再重复触发下载。")
    print(f"  如需查看诊断包，可手动获取：{zip_path}")
else:
    if 'IN_COLAB' in globals() and IN_COLAB and colab_files is not None:
        try:
            colab_files.download(str(zip_path))
            print(f"✓ 已触发诊断包下载：{zip_path.name}")
        except Exception as exc:
            print(f"⚠ 自动下载失败，请手动下载：{zip_path}，原因：{exc}")
    else:
        print(f"ℹ 非 Colab 环境，请手动获取：{zip_path}")

## 配置选项参考

本 Notebook 第 5 步直接使用项目内的 configs/paper_full_cuda.yaml，并通过 scripts/run_paper_full_cuda.py 执行项目输出 workflow。

| 配置项 | 值 | 说明 |
|---------|------|------|
| output entry | scripts/run_paper_full_cuda.py | 论文全机制真实 GPU 运行入口，只负责产出项目输出 |
| cfg | configs/paper_full_cuda.yaml | 仓库内唯一论文全机制配置模板，脚本不再二次定稿 |
| model_id | stabilityai/stable-diffusion-3.5-medium | 真实 SD3.5 模型 |
| device | cuda | 要求 Colab GPU Runtime |
| workflow goal | project outputs only | 仅校验 records / thresholds / evaluation / experiment_matrix 等项目输出 |
| formal acceptance | disabled | 本路径不执行 formal acceptance 工件 |
| signoff | disabled | 本路径不依赖 signoff 工件 |

### 如需调整
- 如需修改模型或参数，请直接在项目配置文件中调整，或新建正式配置文件后在第 5 步改 PROJECT_PAPER_CFG。